In [ ]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 21, IndexedSum, Wedge, MultiEval

Companion notebook to [21_indexed_sum_wedge_multi_eval.md](21_indexed_sum_wedge_multi_eval.md). Three structural Expr nodes underpin everything from Faz 17 onwards. They carry no algebraic content, antisymmetry, distribution, Kronecker contraction, etc. live as engine Definitions in the companion `*_axioms` modules.

## `MultiEval`, multilinear evaluation

`multi_eval(head, *args, alternating=True, slot_kind="vector")`. Slot kind is declarative, `"vector"` (form-on-vectors) or `"covector"` (multivector-on-forms). Arity NOT validated at construction (head's degree may need a registry lookup).

In [ ]:
from jacopy.core.expr import Symbol
from jacopy.core.multi_eval import multi_eval, has_repeated_arg

omega = Symbol("ω")
X, Y  = Symbol("X"), Symbol("Y")

me = multi_eval(omega, X, Y)
print(f"ω(X, Y)        : {me}")
print(f"arity          : {me.arity}")
print(f"alternating    : {me.alternating}")
print(f"slot_kind      : {me.slot_kind}")

swapped, sign = me.swapped(0, 1)
print(f"swap → {swapped}, sign={sign}")

me_repeat = multi_eval(omega, X, X)
print(f"has_repeated_arg : {has_repeated_arg(me_repeat)}")

## `Wedge`, graded-antisymmetric product

Distinct from `Product` (which is non-commutative scalar / operator product). `Wedge.make` flattens, absorbs `0`, drops `1`. Degree-aware identities (`α ∧ α = 0`) live in algorithms layer.

In [ ]:
from jacopy.core.expr import Integer
from jacopy.core.wedge import Wedge

alpha, beta, gamma = Symbol("α"), Symbol("β"), Symbol("γ")

w = Wedge.make(alpha, beta, gamma)
print(f"α ∧ β ∧ γ : {w}")
print(f"absorb 0  : {Wedge.make(alpha, Integer(0), beta)}")
print(f"drop 1    : {Wedge.make(alpha, Integer(1), beta)}")

## `WedgeMultiEvalAlternatingDefinition`

$$(\alpha_1 \wedge \dots \wedge \alpha_p)(X_1, \dots, X_p) = \sum_\sigma \mathrm{sign}(\sigma) \prod_i \alpha_i(X_{\sigma(i)})$$

In [ ]:
from jacopy.calculus.wedge_axioms import WedgeMultiEvalAlternatingDefinition
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.proof.expansion import ExpansionEngine

reg = PropertyRegistry()
for f in (alpha, beta):
    reg.declare(f, Graded(degree=1))
for f in (X, Y):
    reg.declare(f, Graded(degree=0))

w_eval = multi_eval(Wedge.make(alpha, beta), X, Y)
engine = ExpansionEngine([WedgeMultiEvalAlternatingDefinition(registry=reg)])
out, steps = engine.expand(w_eval)
print(f"(α ∧ β)(X, Y) → {out}")
print(f"rule          : {steps[0].rule}")

## `IndexedSum`, α-equivalence as `==`

`indexed_sum(dummy, range_, body)`. Equality compares α-renamed forms via a depth-aware sentinel; nested binders with overlapping names still distinguish correctly.

In [ ]:
from jacopy.calculus.local_frame import FrameIndex, LocalFrame
from jacopy.core.indexed_sum import indexed_sum

i = FrameIndex("i")
j = FrameIndex("j")
frame = LocalFrame("e", dim=3)
T_i = Symbol("T_i")

S   = indexed_sum(i, frame, T_i)
S_j = S.with_dummy(j)
print(f"Σ_i T_i        : {S}")
print(f"Σ_j (renamed)  : {S_j}")
print(f"S == S_j       : {S == S_j}")

## `IndexedSum` axioms

| Rule | Folds |
|---|---|
| `IndexedSumSumDistributeDefinition` | `Σ_i (A + B) → Σ_i A + Σ_i B` |
| `IndexedSumNegPullDefinition` | `Σ_i Neg(X) → Neg(Σ_i X)` |
| `IndexedSumScalarPullDefinition` | `Σ_i (c · X) → c · Σ_i X` (c dummy-free) |
| `IndexedSumKroneckerContractDefinition` | `Σ_i δ_i^j A_i → A_j` |
| `MultiEval/Pairing/ConnectionEvalIndexedSumPushIn` | pull contraction past Σ |

In [ ]:
from jacopy.calculus.indexed_sum_axioms import (
    IndexedSumSumDistributeDefinition,
)
from jacopy.core.expr import Sum

A, B = Symbol("A_i"), Symbol("B_i")
S = indexed_sum(i, frame, Sum(A, B))
engine = ExpansionEngine([IndexedSumSumDistributeDefinition()])
out, steps = engine.expand(S)
print(f"Σ_i (A + B) → {out}")
print(f"rule        : {steps[0].rule}")

## Summary

* `MultiEval(head, *args)`, multilinear evaluation. Slot kind declarative; arity not validated. `swapped(i, j)` carries the alternating sign.
* `Wedge.make(α, β, …)`, graded-antisymmetric product. Smart constructor handles 0/1/associativity; degree-aware `α ∧ α = 0` lives in algorithms layer.
* `IndexedSum(dummy, range_, body)`, bound-index sum with α-equivalence as `==`. `with_dummy(new)` for explicit rename.
* Engine rules in `wedge_axioms` / `indexed_sum_axioms` realise alternating expansion, sum-distribute, scalar-pull, Kronecker contract, push-in past contraction nodes.
* Direct use rare, debugging Cartan / frame proofs is the usual entry point.